# Inspect alphaZero

## imports and args

In [11]:
from Arena import Arena
from KalahGame import KalahGame
from KalahLogic import Board, connect
from utils import *
from MCTS import MCTS  
from KalahPlayers import minmax_tobi, minmax_vince
import time
import multiprocessing
from Coach import Coach
from pytorch.NNet import NNetWrapper
from pytorch.KalahNNet import KalahNNet
import numpy as np
import sys

In [9]:
sys.path.append('..')

In [10]:
args = dotdict({
    'numIters': 1000,
    'numEps': 1,              # Number of complete self-play games to simulate during a new iteration.
    'tempThreshold': 15,        #
    'updateThreshold': 0.6,     # During arena playoff, new neural net will be accepted if threshold or more of games are won.
    'maxlenOfQueue': 200000,    # Number of game examples to train the neural networks.
    'numMCTSSims': 4000,          # Number of games moves for MCTS to simulate.
    'arenaCompare': 40,         # Number of games to play during arena play to determine if new net will be accepted.
    'cpuct': 1,

    'checkpoint': '../best_models',
    'load_model': False,
    'load_folder_file': ('../best_models/','best.pth.tar'),
    'numItersForTrainExamplesHistory': 20,

})

## inspect

In [ ]:
def inspect_train_samples():
    from Coach import Coach 
    c = Coach(KalahGame(), NNetWrapper(KalahGame()), args)
    c.loadTrainExamples()
    return c.trainExamplesHistory

In [15]:
history = inspect_train_samples()

In [30]:
print(len(history)) # different iterations of NN 
print(len(history[0])) # dofferent gamesstates for play with one NN
print(len(history[0][0])) # Board, pi, v
game = 104
print(history[1][game][0]) # Board
print(history[1][game][1]) # pi    
print(history[1][game][2]) # v

20
11570
3
58 | 0 1 1 0 3 1 0 0 |
   | 0 0 0 2 0 0 1 1 | 60

[0, 0, 0, 0, 0, 0, 1, 0]
0.0001


## build New NN

In [ ]:
import sys
sys.path.append('..')
from utils import *

import argparse
import torch
import torch.nn as nn
import torch.nn.functional as F 
import torch.optim as optim


In [3]:
class KalahNNet_conv(nn.Module):
    def __init__(self, game, args):
        # game params
        self.board_x, self.board_y = game.getBoardSize()
        self.action_size = game.getActionSize()
        self.args = args

        super(KalahNNet_conv, self).__init__()
        self.conv1 = nn.Conv2d(1, args.num_channels, 3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(args.num_channels, args.num_channels, 3, stride=1, padding=1)
        self.conv3 = nn.Conv2d(args.num_channels, args.num_channels, 3, stride=1)
        self.conv4 = nn.Conv2d(args.num_channels, args.num_channels, 3, stride=1)

        self.bn1 = nn.BatchNorm2d(args.num_channels)
        self.bn2 = nn.BatchNorm2d(args.num_channels)
        self.bn3 = nn.BatchNorm2d(args.num_channels)
        self.bn4 = nn.BatchNorm2d(args.num_channels)

        self.fc1 = nn.Linear(args.num_channels*(self.board_x-4)*(self.board_y-4), 1024)
        self.fc_bn1 = nn.BatchNorm1d(1024)

        self.fc2 = nn.Linear(1024, 512)
        self.fc_bn2 = nn.BatchNorm1d(512)

        self.fc3 = nn.Linear(512, self.action_size)

        self.fc4 = nn.Linear(512, 1)

    def forward(self, s):
        #                                                           s: batch_size x board_x x board_y
        s = s.view(-1, 1, self.board_x, self.board_y)                # batch_size x 1 x board_x x board_y
        s = F.relu(self.bn1(self.conv1(s)))                          # batch_size x num_channels x board_x x board_y
        s = F.relu(self.bn2(self.conv2(s)))                          # batch_size x num_channels x board_x x board_y
        s = F.relu(self.bn3(self.conv3(s)))                          # batch_size x num_channels x (board_x-2) x (board_y-2)
        s = F.relu(self.bn4(self.conv4(s)))                          # batch_size x num_channels x (board_x-4) x (board_y-4)
        s = s.view(-1, self.args.num_channels*(self.board_x-4)*(self.board_y-4))

        s = F.dropout(F.relu(self.fc_bn1(self.fc1(s))), p=self.args.dropout, training=self.training)  # batch_size x 1024
        s = F.dropout(F.relu(self.fc_bn2(self.fc2(s))), p=self.args.dropout, training=self.training)  # batch_size x 512

        pi = self.fc3(s)                                                                         # batch_size x action_size
        v = self.fc4(s)                                                                          # batch_size x 1

        return F.log_softmax(pi, dim=1), torch.tanh(v)

In [4]:
class KalahNNet_128bn(nn.Module):
    def __init__(self, game, args):
        # game params
        self.board_x, self.board_y = game.getBoardSize()
        self.action_size = game.getActionSize()
        self.args = args

        super(KalahNNet_128bn, self).__init__()
        self.fc1 = nn.Linear(self.board_x * self.board_y, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, self.action_size)
        self.fc4 = nn.Linear(128, 1)

        self.bn1 = nn.BatchNorm1d(128)
        self.bn2 = nn.BatchNorm1d(128)




    def forward(self, s):
        #                                                           s: batch_size x board_x x board_y
        s = s.view(-1, self.board_x * self.board_y)                # batch_size x 1 x (board_x * 2)
        s = F.relu(self.fc1(s))                          # batch_size x num_channels x board_x x board_y
        s = F.relu(self.fc2(s))                          # batch_size x num_channels x board_x x board_y
       
        pi = self.fc3(s)                                                                         # batch_size x action_size
        v = self.fc4(s)                                                                          # batch_size x 1

        return F.log_softmax(pi, dim=1), torch.tanh(v)


## train on previous examples

In [ ]:
import logging
import coloredlogs

nnet_args = dotdict({
    'lr': 0.001,
    'dropout': 0.3,
    'epochs': 10,
    'batch_size': 64,
    'cuda': torch.cuda.is_available(),
    'num_channels': 512,
})

log = logging.getLogger(__name__)

def main():
    g = KalahGame()
    nnet = nn(g)
    nnet.nnet = KalahNNet_128bn(g, nnet_args)
    history = inspect_train_samples()
    nnet.train()


    log.info('Loading the Coach...')
    c = Coach(g, nnet, args)

    if args.load_model:
        log.info("Loading 'trainExamples' from file...")
        c.loadTrainExamples()

    log.info('Starting the learning process 🎉')
    c.learn()

## learn pyTorch

In [38]:
onnet = onn(Game(), args)
params = []
for param in onnet.parameters():
    params.append(param.data)
    print(param)
    
print(len(params))

Parameter containing:
tensor([[-0.1559, -0.1190, -0.0516,  ...,  0.1108,  0.1073,  0.2248],
        [ 0.0339, -0.1593,  0.1329,  ...,  0.2228, -0.1529,  0.1595],
        [-0.0373, -0.1153,  0.1038,  ...,  0.0781, -0.0034, -0.1615],
        ...,
        [ 0.0490,  0.2163, -0.1102,  ..., -0.1571,  0.0343, -0.2309],
        [ 0.1702,  0.2176,  0.0154,  ..., -0.1583,  0.2286,  0.0064],
        [-0.1308, -0.0915,  0.1168,  ...,  0.1448,  0.1096,  0.0360]],
       requires_grad=True)
Parameter containing:
tensor([ 0.0999,  0.0154,  0.1915,  0.0666,  0.0243,  0.0008,  0.2063, -0.1514,
         0.0327,  0.1876, -0.1058, -0.1881, -0.1869, -0.1196, -0.1898, -0.1680,
         0.1471,  0.1397, -0.2104,  0.1592,  0.1317,  0.0502,  0.1532,  0.0377,
        -0.1012,  0.0005,  0.1956,  0.0729,  0.2046, -0.1049,  0.1283, -0.1931,
        -0.0041, -0.0859,  0.1514,  0.0263,  0.1716,  0.0314, -0.1378,  0.0533,
         0.1452,  0.1590, -0.2205,  0.0649,  0.2350,  0.0655, -0.0464,  0.1384,
         0.0957